# Change Blindness

> **Task name:** `Change Blindness`

**Track: Attention** | Can the model detect subtle changes across an interruption?

Humans often fail to notice changes in a scene when the change coincides with a
brief disruption (a blink, a flicker, a cut). This benchmark tests whether LLMs
can detect factual changes between two versions of a passage when separated by
a disruptor paragraph.

**Metrics:**
- Detection rate by change magnitude (minor vs. major)
- Detection rate by disruptor length (0, 1, 3 paragraphs)
- False alarm rate (reporting non-existent changes)
- Specificity (identifying exactly what changed)

In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd
import numpy as np
import json
import re
import random

print(f"Available models: {list(kbench.llms.keys())}")

In [ ]:
def strip_thinking(text: str) -> str:
    if "</think>" in text:
        return text.split("</think>", 1)[1].strip()
    return text.strip()

def normalize(s: str) -> str:
    return re.sub(r"\s+", " ", s.lower().strip())

In [ ]:
# --- Passages with controlled changes ---
# Two categories:
#   code-development: actual code snippets with subtle modifications
#   code-task: ticket/spec descriptions of work to implement

PASSAGES = [
    {
        "id": "retry_logic",
        "category": "code-development",
        "original": """def retry_request(url, max_retries=3, backoff=2.0):
    for attempt in range(max_retries):
        try:
            resp = requests.get(url, timeout=10)
            resp.raise_for_status()
            return resp.json()
        except requests.RequestException as e:
            if attempt == max_retries - 1:
                raise
            wait = backoff ** attempt
            time.sleep(wait)
    return None""",
        "minor_change": """def retry_request(url, max_retries=5, backoff=2.0):
    for attempt in range(max_retries):
        try:
            resp = requests.get(url, timeout=10)
            resp.raise_for_status()
            return resp.json()
        except requests.RequestException as e:
            if attempt == max_retries - 1:
                raise
            wait = backoff ** attempt
            time.sleep(wait)
    return None""",
        "minor_detail": "max_retries=3 changed to max_retries=5",
        "major_change": """def retry_request(url, max_retries=3, backoff=2.0):
    for attempt in range(max_retries):
        try:
            resp = requests.get(url, timeout=10)
            resp.raise_for_status()
            return resp.json()
        except requests.RequestException as e:
            if attempt == max_retries - 1:
                return {}
            wait = backoff ** attempt
            time.sleep(wait)
    return None""",
        "major_detail": "raise changed to return {} — silently swallows the error instead of raising",
    },
    {
        "id": "cache_decorator",
        "category": "code-development",
        "original": """class LRUCache:
    def __init__(self, capacity=128):
        self.capacity = capacity
        self.cache = OrderedDict()

    def get(self, key):
        if key not in self.cache:
            return None
        self.cache.move_to_end(key)
        return self.cache[key]

    def put(self, key, value):
        if key in self.cache:
            self.cache.move_to_end(key)
        self.cache[key] = value
        if len(self.cache) > self.capacity:
            self.cache.popitem(last=False)""",
        "minor_change": """class LRUCache:
    def __init__(self, capacity=256):
        self.capacity = capacity
        self.cache = OrderedDict()

    def get(self, key):
        if key not in self.cache:
            return None
        self.cache.move_to_end(key)
        return self.cache[key]

    def put(self, key, value):
        if key in self.cache:
            self.cache.move_to_end(key)
        self.cache[key] = value
        if len(self.cache) > self.capacity:
            self.cache.popitem(last=False)""",
        "minor_detail": "capacity=128 changed to capacity=256",
        "major_change": """class LRUCache:
    def __init__(self, capacity=128):
        self.capacity = capacity
        self.cache = OrderedDict()

    def get(self, key):
        if key not in self.cache:
            return None
        self.cache.move_to_end(key)
        return self.cache[key]

    def put(self, key, value):
        if key in self.cache:
            self.cache.move_to_end(key)
        self.cache[key] = value
        if len(self.cache) > self.capacity:
            self.cache.popitem(last=True)""",
        "major_detail": "popitem(last=False) changed to popitem(last=True) — evicts most recently used instead of least recently used",
    },
    {
        "id": "rate_limiter",
        "category": "code-development",
        "original": """async def rate_limit(request, max_rpm=60):
    key = f"rate:{request.client_ip}"
    count = await redis.incr(key)
    if count == 1:
        await redis.expire(key, 60)
    if count > max_rpm:
        return JSONResponse(
            status_code=429,
            content={"error": "Rate limit exceeded"}
        )
    return await call_next(request)""",
        "minor_change": """async def rate_limit(request, max_rpm=120):
    key = f"rate:{request.client_ip}"
    count = await redis.incr(key)
    if count == 1:
        await redis.expire(key, 60)
    if count > max_rpm:
        return JSONResponse(
            status_code=429,
            content={"error": "Rate limit exceeded"}
        )
    return await call_next(request)""",
        "minor_detail": "max_rpm=60 changed to max_rpm=120",
        "major_change": """async def rate_limit(request, max_rpm=60):
    key = f"rate:{request.client_ip}"
    count = await redis.incr(key)
    if count == 1:
        await redis.expire(key, 60)
    if count > max_rpm:
        await redis.delete(key)
        return await call_next(request)
    return await call_next(request)""",
        "major_detail": "returns 429 error changed to deletes the key and lets the request through — rate limit never actually blocks",
    },
    {
        "id": "search_ticket",
        "category": "code-task",
        "original": """TICKET: SEARCH-142 — Add fuzzy matching to product search
Priority: P1
Assignee: @maria

Requirements:
- Use Levenshtein distance with a threshold of 2 edits
- Only apply fuzzy matching when exact match returns < 3 results
- Index the top 50,000 products by name and SKU
- Results must return within 200ms at p95
- Write integration tests against the staging Elasticsearch cluster
- Add a feature flag "fuzzy_search_enabled" defaulting to OFF""",
        "minor_change": """TICKET: SEARCH-142 — Add fuzzy matching to product search
Priority: P1
Assignee: @maria

Requirements:
- Use Levenshtein distance with a threshold of 2 edits
- Only apply fuzzy matching when exact match returns < 3 results
- Index the top 50,000 products by name and SKU
- Results must return within 500ms at p95
- Write integration tests against the staging Elasticsearch cluster
- Add a feature flag "fuzzy_search_enabled" defaulting to OFF""",
        "minor_detail": "200ms at p95 changed to 500ms at p95",
        "major_change": """TICKET: SEARCH-142 — Add fuzzy matching to product search
Priority: P1
Assignee: @maria

Requirements:
- Use Levenshtein distance with a threshold of 2 edits
- Only apply fuzzy matching when exact match returns < 3 results
- Index the top 50,000 products by name and SKU
- Results must return within 200ms at p95
- Write integration tests against the staging Elasticsearch cluster
- Ship directly to production with no feature flag""",
        "major_detail": "feature flag defaulting to OFF changed to ship directly to production with no feature flag",
    },
    {
        "id": "migration_ticket",
        "category": "code-task",
        "original": """TICKET: DATA-87 — Migrate user profiles from MongoDB to PostgreSQL
Priority: P2
Assignee: @james

Requirements:
- Write a migration script that processes users in batches of 1,000
- Preserve all existing timestamps (created_at, updated_at)
- Map the nested address subdocument to a separate addresses table
- Run a validation pass comparing row counts and checksums
- Keep MongoDB as read-only fallback for 2 weeks post-migration
- Schedule the migration for the Sunday maintenance window (2am-6am EST)""",
        "minor_change": """TICKET: DATA-87 — Migrate user profiles from MongoDB to PostgreSQL
Priority: P2
Assignee: @james

Requirements:
- Write a migration script that processes users in batches of 5,000
- Preserve all existing timestamps (created_at, updated_at)
- Map the nested address subdocument to a separate addresses table
- Run a validation pass comparing row counts and checksums
- Keep MongoDB as read-only fallback for 2 weeks post-migration
- Schedule the migration for the Sunday maintenance window (2am-6am EST)""",
        "minor_detail": "batches of 1,000 changed to batches of 5,000",
        "major_change": """TICKET: DATA-87 — Migrate user profiles from MongoDB to PostgreSQL
Priority: P2
Assignee: @james

Requirements:
- Write a migration script that processes users in batches of 1,000
- Preserve all existing timestamps (created_at, updated_at)
- Map the nested address subdocument to a separate addresses table
- Run a validation pass comparing row counts and checksums
- Drop MongoDB immediately after migration with no fallback period
- Schedule the migration for the Sunday maintenance window (2am-6am EST)""",
        "major_detail": "keep MongoDB as read-only fallback for 2 weeks changed to drop MongoDB immediately with no fallback",
    },
]

DISRUPTORS = [
    "STANDUP NOTE: The team discussed switching from biweekly to weekly sprint planning. No decision was made but @alex will send out a poll. Also, the snack budget for Q3 was approved at $150/month.",
    "SLACK: @devops posted that the VPN certificate expires next Thursday. Everyone should re-download their config from the internal portal before then. The new certificate uses 4096-bit RSA keys.",
    "WIKI UPDATE: The onboarding doc was updated to include the new GitHub org structure. Repos are now split into platform/, services/, and tools/ top-level directories instead of the old flat layout.",
]

CONFIDENCE_SUFFIX = (
    "\n\nAfter stating the change (or NO CHANGE), rate your confidence on a scale of 1-5:\n"
    "1 = Very unsure, 2 = Somewhat unsure, 3 = Neutral, 4 = Somewhat confident, 5 = Very confident\n\n"
    "Format:\n"
    "Change: <your answer>\n"
    "Confidence: <1-5>"
)

random.seed(123)
data = []
task_id = 0

for passage in PASSAGES:
    for change_type in ["minor", "major"]:
        changed_text = passage[f"{change_type}_change"]
        expected_detail = passage[f"{change_type}_detail"]

        for disruptor_count in [0, 1, 3]:
            disruptor_text = ""
            if disruptor_count > 0:
                selected = random.sample(DISRUPTORS, min(disruptor_count, len(DISRUPTORS)))
                if disruptor_count > len(DISRUPTORS):
                    selected = selected * (disruptor_count // len(DISRUPTORS) + 1)
                    selected = selected[:disruptor_count]
                disruptor_text = "\n\n".join(selected)

            if disruptor_count == 0:
                prompt = (
                    "Read Version A and Version B carefully. "
                    "Identify what changed between them.\n\n"
                    f"=== VERSION A ===\n{passage['original']}\n\n"
                    f"=== VERSION B ===\n{changed_text}\n\n"
                    "What specific detail changed between Version A and Version B? "
                    "State the change precisely."
                    + CONFIDENCE_SUFFIX
                )
            else:
                prompt = (
                    "Read Version A, then the interlude, then Version B carefully. "
                    "Identify what changed between the two versions.\n\n"
                    f"=== VERSION A ===\n{passage['original']}\n\n"
                    f"=== INTERLUDE ===\n{disruptor_text}\n\n"
                    f"=== VERSION B ===\n{changed_text}\n\n"
                    "What specific detail changed between Version A and Version B? "
                    "Ignore the interlude — it is unrelated. "
                    "State the change precisely."
                    + CONFIDENCE_SUFFIX
                )

            data.append({
                "task_id": task_id,
                "prompt": prompt,
                "expected": expected_detail,
                "passage_id": passage["id"],
                "category": passage["category"],
                "change_type": change_type,
                "disruptor_count": disruptor_count,
            })
            task_id += 1

    # Control: no-change condition
    for disruptor_count in [0, 1, 3]:
        disruptor_text = ""
        if disruptor_count > 0:
            selected = random.sample(DISRUPTORS, min(disruptor_count, len(DISRUPTORS)))
            if disruptor_count > len(DISRUPTORS):
                selected = selected * (disruptor_count // len(DISRUPTORS) + 1)
                selected = selected[:disruptor_count]
            disruptor_text = "\n\n".join(selected)

        if disruptor_count == 0:
            prompt = (
                "Read Version A and Version B carefully. "
                "Identify what changed between them. "
                "If nothing changed, respond with exactly: NO CHANGE\n\n"
                f"=== VERSION A ===\n{passage['original']}\n\n"
                f"=== VERSION B ===\n{passage['original']}\n\n"
                "What specific detail changed between Version A and Version B? "
                "If they are identical, respond with exactly: NO CHANGE"
                + CONFIDENCE_SUFFIX
            )
        else:
            prompt = (
                "Read Version A, then the interlude, then Version B carefully. "
                "Identify what changed between the two versions. "
                "If nothing changed, respond with exactly: NO CHANGE\n\n"
                f"=== VERSION A ===\n{passage['original']}\n\n"
                f"=== INTERLUDE ===\n{disruptor_text}\n\n"
                f"=== VERSION B ===\n{passage['original']}\n\n"
                "What specific detail changed between Version A and Version B? "
                "Ignore the interlude. "
                "If they are identical, respond with exactly: NO CHANGE"
                + CONFIDENCE_SUFFIX
            )

        data.append({
            "task_id": task_id,
            "prompt": prompt,
            "expected": "NO CHANGE",
            "passage_id": passage["id"],
            "category": passage["category"],
            "change_type": "none",
            "disruptor_count": disruptor_count,
        })
        task_id += 1

df_all = pd.DataFrame(data)
print(f"Generated {len(df_all)} items")
print(f"By category: {dict(df_all['category'].value_counts())}")
print(f"By change type: {dict(df_all['change_type'].value_counts())}")
print(f"By disruptor count: {dict(df_all['disruptor_count'].value_counts())}")

In [ ]:
# --- Dataset overview ---
print("=== code-development (actual code with subtle changes) ===\n")
for p in PASSAGES:
    if p["category"] != "code-development":
        continue
    print(f"[{p['id']}]")
    print(f"  Minor: {p['minor_detail']}")
    print(f"  Major: {p['major_detail']}")
    print()

print("=== code-task (tickets/specs with requirement changes) ===\n")
for p in PASSAGES:
    if p["category"] != "code-task":
        continue
    print(f"[{p['id']}]")
    print(f"  Minor: {p['minor_detail']}")
    print(f"  Major: {p['major_detail']}")
    print()

print("=== Design ===")
print(f"  {len(PASSAGES)} passages × 3 change types × 3 disruptor levels = {len(df_all)} items")
print(f"  code-development: {sum(1 for p in PASSAGES if p['category'] == 'code-development')} passages (code snippets)")
print(f"  code-task: {sum(1 for p in PASSAGES if p['category'] == 'code-task')} passages (tickets/specs)")

## Task Definition

In [ ]:
@kbench.task(
    name="change_blindness",
    description="Detect subtle factual changes between two passage versions separated by a disruptor paragraph"
)
def change_blindness(
    llm,
    prompt: str,
    expected: str,
    change_type: str,
    disruptor_count: int,
) -> bool:
    response = llm.prompt(prompt)
    response = strip_thinking(response)
    resp_norm = normalize(response)

    if expected == "NO CHANGE":
        return "no change" in resp_norm

    # Extract key values from expected (e.g., "500 requests per minute changed to 200 requests per minute")
    # Split on common transition phrases
    parts = re.split(r"changed to|changed from|→|->|replaced with|replaced by", expected.lower())
    if len(parts) >= 2:
        old_part = normalize(parts[0])
        new_part = normalize(parts[1])
        # Extract just the numbers/key terms from each part
        old_numbers = re.findall(r'\d[\d,]*', old_part)
        new_numbers = re.findall(r'\d[\d,]*', new_part)
        # Check if both key numbers appear in response
        if old_numbers and new_numbers:
            old_found = any(n.replace(',', '') in resp_norm.replace(',', '') for n in old_numbers)
            new_found = any(n.replace(',', '') in resp_norm.replace(',', '') for n in new_numbers)
            if old_found and new_found:
                return True
        # Fall back to checking key phrases (3+ word chunks)
        old_words = [w for w in old_part.split() if len(w) > 2]
        new_words = [w for w in new_part.split() if len(w) > 2]
        # At least half the meaningful words from each side should appear
        if old_words and new_words:
            old_match = sum(1 for w in old_words if w in resp_norm) / len(old_words)
            new_match = sum(1 for w in new_words if w in resp_norm) / len(new_words)
            return old_match >= 0.5 and new_match >= 0.5

    # Last resort: full substring
    return normalize(expected) in resp_norm

## Run Evaluation

In [ ]:
eval_df = df_all[["prompt", "expected", "change_type", "disruptor_count"]].copy()

print(f"Running {len(eval_df)} items with kbench.llm...")
runs = change_blindness.evaluate(
    llm=[kbench.llm],
    evaluation_data=eval_df,
    n_jobs=2,
    timeout=120,
    max_attempts=2,
)
results = runs.as_dataframe()
results = results.merge(
    df_all[["prompt", "passage_id", "change_type", "disruptor_count"]],
    on="prompt", how="left", suffixes=("", "_meta")
)
print(f"Collected {len(results)} results")

## Results & Analysis

In [ ]:
# Convert bool result to float
results["correct"] = results["result"].apply(lambda r: float(r) if isinstance(r, bool) else float(r))

print("=== Detection Rate by Change Type × Disruptor Count ===")
pivot = results.groupby(["change_type", "disruptor_count"])["correct"].mean().unstack()
print(pivot.to_string(float_format="%.2f"))

print("\n=== Overall by Change Type ===")
print(results.groupby("change_type")["correct"].agg(["mean", "count"]).to_string(float_format="%.2f"))

# False alarm rate
no_change = results[results["change_type"] == "none"]
false_alarm = 1 - no_change["correct"].mean() if len(no_change) > 0 else 0
print(f"\nFalse alarm rate (saying something changed when nothing did): {false_alarm:.2%}")

# Change detection drop with disruptors
for ct in ["minor", "major"]:
    ct_data = results[results["change_type"] == ct]
    baseline = ct_data[ct_data["disruptor_count"] == 0]["correct"].mean()
    with_3 = ct_data[ct_data["disruptor_count"] == 3]["correct"].mean()
    print(f"{ct} change: baseline={baseline:.2%}, with 3 disruptors={with_3:.2%}, drop={baseline-with_3:.2%}")

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "matplotlib"], check=True)
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))
for ct, marker, color in [("minor", "s", "orange"), ("major", "^", "blue"), ("none", "o", "red")]:
    subset = results[results["change_type"] == ct]
    curve = subset.groupby("disruptor_count")["correct"].mean()
    label = f"{ct} change" if ct != "none" else "no change (1 - false alarm)"
    vals = curve.values if ct != "none" else 1 - curve.values
    ax.plot(curve.index, vals, f"{marker}-", label=label, linewidth=2, markersize=8, color=color)

ax.set_xlabel("Number of Disruptor Paragraphs")
ax.set_ylabel("Detection Rate")
ax.set_title("Change Blindness: Detection Rate by Disruption Level")
ax.set_xticks([0, 1, 3])
ax.set_ylim(-0.05, 1.05)
ax.legend()
plt.tight_layout()
plt.savefig("change_blindness_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved to change_blindness_results.png")

## Confidence Calibration Analysis

Does the model know when it's wrong? We parse the confidence ratings (1-5) from responses and compare against actual accuracy to measure metacognitive calibration.

In [ ]:
# Parse confidence from model responses
def extract_confidence(response_text: str) -> int | None:
    """Extract confidence rating (1-5) from response."""
    text = strip_thinking(str(response_text))
    # Look for "Confidence: N" pattern
    match = re.search(r"[Cc]onfidence\s*[:=]\s*(\d)", text)
    if match:
        val = int(match.group(1))
        return val if 1 <= val <= 5 else None
    return None

# Extract confidence from raw responses
if "response" in results.columns:
    results["confidence"] = results["response"].apply(extract_confidence)
else:
    # Try to get from the evaluation run
    results["confidence"] = None
    print("Note: Raw responses not available in results DataFrame.")
    print("Confidence analysis requires the raw response text.")

conf_results = results.dropna(subset=["confidence"]).copy()
conf_results["confidence"] = conf_results["confidence"].astype(int)

if len(conf_results) > 0:
    print(f"=== Confidence Calibration ({len(conf_results)} items with confidence) ===\n")
    
    # Calibration table: for each confidence level, what's the actual accuracy?
    print("Confidence | Accuracy | Count | Calibration Gap")
    print("-" * 55)
    
    calibration_data = []
    for conf_level in range(1, 6):
        subset = conf_results[conf_results["confidence"] == conf_level]
        if len(subset) > 0:
            acc = subset["correct"].mean()
            expected_acc = conf_level / 5.0  # Linear mapping: 1→0.2, 5→1.0
            gap = acc - expected_acc
            calibration_data.append({
                "confidence": conf_level,
                "accuracy": acc,
                "expected": expected_acc,
                "count": len(subset),
                "gap": gap,
            })
            print(f"    {conf_level}      |  {acc:.2%}  |  {len(subset):3d}  | {gap:+.2%}")
    
    cal_df = pd.DataFrame(calibration_data)
    
    # Expected Calibration Error (ECE)
    if len(cal_df) > 0:
        total = cal_df["count"].sum()
        ece = sum(row["count"] / total * abs(row["gap"]) for _, row in cal_df.iterrows())
        print(f"\nExpected Calibration Error (ECE): {ece:.4f}")
    
    # Overconfidence: wrong answers with confidence 4-5
    wrong_high_conf = conf_results[(conf_results["correct"] == 0) & (conf_results["confidence"] >= 4)]
    total_wrong = conf_results[conf_results["correct"] == 0]
    if len(total_wrong) > 0:
        overconf_rate = len(wrong_high_conf) / len(total_wrong)
        print(f"Overconfidence rate (wrong + conf>=4): {overconf_rate:.2%} ({len(wrong_high_conf)}/{len(total_wrong)})")
    
    # Underconfidence: correct answers with confidence 1-2
    correct_low_conf = conf_results[(conf_results["correct"] == 1) & (conf_results["confidence"] <= 2)]
    total_correct = conf_results[conf_results["correct"] == 1]
    if len(total_correct) > 0:
        underconf_rate = len(correct_low_conf) / len(total_correct)
        print(f"Underconfidence rate (correct + conf<=2): {underconf_rate:.2%} ({len(correct_low_conf)}/{len(total_correct)})")
    
    # Calibration by change type
    print("\n=== Confidence by Change Type ===")
    print(conf_results.groupby("change_type")[["confidence", "correct"]].mean().to_string(float_format="%.2f"))
    
    # Plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Reliability diagram
    if len(cal_df) > 0:
        ax1.bar(cal_df["confidence"], cal_df["accuracy"], width=0.6, color="steelblue", 
                edgecolor="black", linewidth=0.5, label="Actual accuracy")
        ax1.plot([1, 5], [0.2, 1.0], "r--", linewidth=2, label="Perfect calibration")
        ax1.set_xlabel("Confidence Level")
        ax1.set_ylabel("Actual Accuracy")
        ax1.set_title("Reliability Diagram")
        ax1.set_xticks([1, 2, 3, 4, 5])
        ax1.set_ylim(0, 1.1)
        ax1.legend()
    
    # Confidence distribution for correct vs incorrect
    for label, subset, color in [
        ("Correct", conf_results[conf_results["correct"] == 1], "seagreen"),
        ("Incorrect", conf_results[conf_results["correct"] == 0], "tomato"),
    ]:
        if len(subset) > 0:
            counts = subset["confidence"].value_counts().reindex(range(1, 6), fill_value=0)
            offset = 0.2 if label == "Correct" else -0.2
            ax2.bar(counts.index + offset, counts.values, width=0.35, color=color,
                    edgecolor="black", linewidth=0.5, label=label)
    
    ax2.set_xlabel("Confidence Level")
    ax2.set_ylabel("Count")
    ax2.set_title("Confidence Distribution: Correct vs Incorrect")
    ax2.set_xticks([1, 2, 3, 4, 5])
    ax2.legend()
    
    plt.tight_layout()
    plt.savefig("change_blindness_calibration.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Calibration plot saved to change_blindness_calibration.png")
else:
    print("No confidence data available — model may not have followed the confidence format.")